In [1]:
from pathlib import Path
import re
import pandas as pd

# Folder containing the importance-magnitude CSV files
input_folder = Path(".")

# Match files such as:
filename_pattern = re.compile(
    r"^indiv_importance_magnitude_cnn1_HT11_(\d+)\.csv$"
)

# Detect matching files
matching_files = []

for file_path in input_folder.glob(
    "indiv_importance_magnitude_cnn1_HT11_*.csv"
):
    match = filename_pattern.match(file_path.name)

    if match:
        model_number = int(match.group(1))
        matching_files.append((model_number, file_path))

# Sort by model number
matching_files.sort(key=lambda x: x[0])

print(f"Detected {len(matching_files)} importance-magnitude files:")

for model_number, file_path in matching_files:
    print(f"Model {model_number}: {file_path.name}")

if len(matching_files) == 0:
    raise FileNotFoundError(
        "No matching importance-magnitude CSV files were found."
    )

if len(matching_files) != 10:
    print(
        f"\nWarning: Expected 10 files, but detected "
        f"{len(matching_files)} files."
    )


# Read and combine all files
all_importance_tables = []

for model_number, file_path in matching_files:

    df = pd.read_csv(file_path)

    # Check that the expected columns exist
    required_columns = {"Feature", "Importance Magnitude"}

    if not required_columns.issubset(df.columns):
        raise ValueError(
            f"{file_path.name} does not contain the required columns: "
            f"'Feature' and 'Importance Magnitude'."
        )

    # Keep only the required columns
    df = df[["Feature", "Importance Magnitude"]].copy()

    # Convert importance values to numeric
    df["Importance Magnitude"] = pd.to_numeric(
        df["Importance Magnitude"],
        errors="coerce"
    )

    # Add model number for tracking
    df["Model Number"] = model_number

    all_importance_tables.append(df)

# Combine all model tables
combined_df = pd.concat(
    all_importance_tables,
    ignore_index=True
)

# Calculate average importance magnitude for each feature
average_importance = (
    combined_df
    .groupby("Feature", as_index=False)
    .agg(
        Average_Importance_Magnitude=(
            "Importance Magnitude",
            "mean"
        ),
        Number_of_Models=(
            "Importance Magnitude",
            "count"
        )
    )
)

# Sort from highest to lowest average importance
average_importance = average_importance.sort_values(
    by="Average_Importance_Magnitude",
    ascending=False,
    ignore_index=True
)

# Save the final table
output_file = (
    input_folder /
    "indiv_average_importance_magnitude_cnn1_HT11.csv"
)

average_importance.to_csv(output_file, index=False)

print(f"\nSaved: {output_file.name}")
print("\nAverage importance magnitude table:")
print(average_importance)

Detected 10 importance-magnitude files:
Model 11: indiv_importance_magnitude_cnn1_HT11_11.csv
Model 13: indiv_importance_magnitude_cnn1_HT11_13.csv
Model 14: indiv_importance_magnitude_cnn1_HT11_14.csv
Model 30: indiv_importance_magnitude_cnn1_HT11_30.csv
Model 31: indiv_importance_magnitude_cnn1_HT11_31.csv
Model 33: indiv_importance_magnitude_cnn1_HT11_33.csv
Model 76: indiv_importance_magnitude_cnn1_HT11_76.csv
Model 77: indiv_importance_magnitude_cnn1_HT11_77.csv
Model 88: indiv_importance_magnitude_cnn1_HT11_88.csv
Model 95: indiv_importance_magnitude_cnn1_HT11_95.csv

Saved: indiv_average_importance_magnitude_cnn1_HT11.csv

Average importance magnitude table:
     Feature  Average_Importance_Magnitude  Number_of_Models
0    Pos40_G                      0.007442                10
1    Pos22_C                      0.005271                10
2    Pos22_T                      0.004244                10
3    Pos27_T                      0.004178                10
4    Pos26_T         

In [3]:
import pandas as pd

# Read the average importance-magnitude CSV file
df = pd.read_csv("indiv_average_importance_magnitude_cnn1_HT11.csv")

# Extract position and base from feature names
extracted = df["Feature"].str.extract(r"^Pos(\d+)_([ACGT])$")

df["Position"] = pd.to_numeric(extracted[0], errors="coerce")
df["Base"] = extracted[1]

# Ensure importance magnitudes are numeric
df["Average_Importance_Magnitude"] = pd.to_numeric(
    df["Average_Importance_Magnitude"],
    errors="coerce"
)

# Remove rows that don't match the expected format
df = df.dropna(
    subset=["Position", "Base", "Average_Importance_Magnitude"]
).copy()

df["Position"] = df["Position"].astype(int)

# Define windows (using actual positions)
windows = {
    "First_6": (21, 26),    # Pos21–Pos26
    "Middle_8": (27, 34),   # Pos27–Pos34
    "Last_6": (35, 40)      # Pos35–Pos40
}

bases = ["A", "C", "G", "T"]

results = []

for base in bases:
    base_df = df[df["Base"] == base]

    row = {"Base": base}

    for window_name, (start, end) in windows.items():
        avg = base_df.loc[
            base_df["Position"].between(start, end),
            "Average_Importance_Magnitude"
        ].mean()

        row[window_name] = avg

    results.append(row)

# Convert to DataFrame
result_df = pd.DataFrame(results)

# Save as CSV
result_df.to_csv(
    "indiv_average_importance_magnitude_by_Base_position_range_cnn1_HT11.csv",
    index=False
)

print(result_df)

  Base   First_6  Middle_8    Last_6
0    A  0.001461  0.001266  0.001306
1    C  0.002969  0.002316  0.002133
2    G  0.002204  0.001880  0.003410
3    T  0.003169  0.002788  0.002324
